<a href="https://colab.research.google.com/github/equlo/triangularaveragefuzzyfilter_finalyearproject/blob/main/Final_Year_Project_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# LOW LIGHT IMAGE ENHANCEMENT USING
# TRIANGULAR FUZZY FILTER + CLAHE + SHARPENING
# =====================================================

# Install if needed
# !pip install opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# IMAGE UPLOAD
# =====================================================

from google.colab import files

uploaded = files.upload()

image_name = list(uploaded.keys())[0]

print("Image Selected :", image_name)

# =====================================================
# LOAD IMAGE
# =====================================================

img = cv2.imread(image_name)

if img is None:
    raise ValueError("Image could not be loaded.")

# =====================================================
# ENTROPY FUNCTION
# =====================================================

def calculate_entropy(image):

    hist = cv2.calcHist([image],[0],None,[256],[0,256])

    hist = hist.ravel()

    hist = hist / hist.sum()

    hist = hist[hist > 0]

    return -np.sum(hist * np.log2(hist))

# =====================================================
# CONVERT TO HSV
# =====================================================

hsv = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2HSV
)

h, s, v = cv2.split(hsv)

# =====================================================
# NORMALIZATION
# =====================================================

v = v.astype(np.float32) / 255.0

# =====================================================
# TRIANGULAR FUZZY MEMBERSHIP FUNCTION
# =====================================================

a = 0.0
b = 0.40
c = 1.0

membership = np.maximum(
    np.minimum(
        (v - a) / (b - a),
        (c - v) / (c - b)
    ),
    0
)

# =====================================================
# FUZZY ENHANCEMENT
# =====================================================

k = 0.8

enhanced_v = v + k * membership

enhanced_v = np.clip(
    enhanced_v,
    0,
    1
)

enhanced_v = (
    enhanced_v * 255
).astype(np.uint8)

# =====================================================
# CLAHE CONTRAST ENHANCEMENT
# =====================================================

clahe = cv2.createCLAHE(
    clipLimit=3.0,
    tileGridSize=(8,8)
)

enhanced_v = clahe.apply(
    enhanced_v
)

# =====================================================
# MERGE HSV CHANNELS
# =====================================================

enhanced_hsv = cv2.merge(
    [h, s, enhanced_v]
)

result = cv2.cvtColor(
    enhanced_hsv,
    cv2.COLOR_HSV2BGR
)

# =====================================================
# SHARPENING FILTER
# =====================================================

kernel = np.array([
    [0,-1,0],
    [-1,5,-1],
    [0,-1,0]
])

sharpened = cv2.filter2D(
    result,
    -1,
    kernel
)

# =====================================================
# ENTROPY CALCULATION
# =====================================================

original_gray = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2GRAY
)

enhanced_gray = cv2.cvtColor(
    sharpened,
    cv2.COLOR_BGR2GRAY
)

original_entropy = calculate_entropy(
    original_gray
)

enhanced_entropy = calculate_entropy(
    enhanced_gray
)

# =====================================================
# PRINT RESULTS
# =====================================================

print("\n==============================")
print("RESULTS")
print("==============================")

print(
    "Original Entropy :",
    round(original_entropy,4)
)

print(
    "Enhanced Entropy:",
    round(enhanced_entropy,4)
)

# =====================================================
# SAVE IMAGE
# =====================================================

output_name = "enhanced_output.jpg"

cv2.imwrite(
    output_name,
    sharpened
)

print("\nEnhanced image saved as:")
print(output_name)

# =====================================================
# DISPLAY RESULTS
# =====================================================

plt.figure(figsize=(15,7))

plt.subplot(1,2,1)

plt.imshow(
    cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )
)

plt.title("Original Image")
plt.axis("off")

plt.subplot(1,2,2)

plt.imshow(
    cv2.cvtColor(
        sharpened,
        cv2.COLOR_BGR2RGB
    )
)

plt.title("Enhanced Image")
plt.axis("off")

plt.show()